# LLMOps Pipeline — Phi-3-mini Medical QA Fine-tuning

**Runtime**: Make sure you have **T4 GPU** selected
Runtime → Change runtime type → T4 GPU

This notebook fine-tunes `microsoft/Phi-3-mini-4k-instruct` on medical QA data using **LoRA (QLoRA)**.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers peft trl bitsandbytes accelerate datasets mlflow huggingface-hub evaluate rouge-score nltk

In [ ]:
# Cell 2: Configuration
import os
from google.colab import userdata

# Set your HuggingFace token as a Colab secret named HF_TOKEN
HF_TOKEN = userdata.get('HF_TOKEN')
BASE_MODEL = 'microsoft/Phi-3-mini-4k-instruct'
FINETUNED_REPO = 'djism/phi3-medical-qa'
LORA_R = 16
LORA_ALPHA = 32
NUM_EPOCHS = 3
LEARNING_RATE = 0.0002
MAX_SEQ_LENGTH = 512

print(f'Base model: {BASE_MODEL}')
print(f'Fine-tuned repo: {FINETUNED_REPO}')

In [ ]:
# Cell 3: Load and prepare dataset
from datasets import load_dataset
import json, random

print('Loading medical QA dataset...')
dataset = load_dataset('medalpaca/medical_meadow_medqa', split='train')
print(f'Loaded {len(dataset)} samples')

def format_sample(sample):
    instruction = sample.get('instruction', '')
    output = sample.get('output', '')
    return {'text': f'<|user|>\n{instruction}<|end|>\n<|assistant|>\n{output}<|end|>'}

# Filter and format
samples = [s for s in dataset if len(s.get('instruction','')) > 20 and len(s.get('output','')) > 20]
random.shuffle(samples)
samples = samples[:2000]  # Keep manageable

from datasets import Dataset
formatted = Dataset.from_list([format_sample(s) for s in samples])

# Split 85/15
split = formatted.train_test_split(test_size=0.15, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# Cell 4: Load model with 4-bit quantization (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN
)
print('Base model loaded!')

In [ ]:
# Cell 5: Apply LoRA
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type=TaskType.CAUSAL_LM,
    bias='none'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('LoRA applied!')

In [ ]:
# Cell 6: Train
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./phi3-medical-qa',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=True,
    logging_steps=10,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 7: Save and push to HuggingFace Hub
trainer.save_model('./phi3-medical-qa')
tokenizer.save_pretrained('./phi3-medical-qa')

model.push_to_hub(FINETUNED_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(FINETUNED_REPO, token=HF_TOKEN)

print(f'Model pushed to: https://huggingface.co/{FINETUNED_REPO}')